# 05 — Separar e exportar para revisão

Objetivo: criar treino, validação e teste sem que um problema-base apareça em mais de um conjunto, e preparar uma amostra CSV para professores. Entrada: `data/interim/diagnosticos_dpo.jsonl`. Saídas: `data/final/train.jsonl`, `val.jsonl`, `test.jsonl` e `amostra_revisao.csv`. A divisão é feita por `base_id`, com semente fixa.

In [1]:
# Opcional: %pip install pandas
import json
import random
from pathlib import Path
from typing import Any

import pandas as pd

def localizar_projeto() -> Path:
    '''Localiza o projeto independente do diretório do kernel.'''
    for base in (Path.cwd(), *Path.cwd().parents):
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists(): return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists(): return base
    raise FileNotFoundError('Configuração compartilhada não encontrada')

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Escapa separadores Unicode para manter um registro por linha física.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
random.seed(CONFIG['seed'])
ENTRADA = ROOT / 'data' / 'interim' / 'diagnosticos_dpo.jsonl'
PASTA_FINAL = ROOT / 'data' / 'final'
if not ENTRADA.exists(): raise FileNotFoundError(f'Execute primeiro o notebook 04: {ENTRADA}')
registros: list[dict[str, Any]] = [json.loads(l) for l in ENTRADA.read_text(encoding='utf-8').splitlines() if l]
if not registros: raise ValueError('Não há registros para dividir')
grupos = sorted({r['base_id'] for r in registros})
random.shuffle(grupos)
n = len(grupos)
if n >= 3:
    corte_treino = max(1, min(n - 2, int(round(n * 0.8))))
    corte_validacao = max(corte_treino + 1, min(n - 1, int(round(n * 0.9))))
else:
    corte_treino = max(1, n - 1)
    corte_validacao = n
grupos_por_split = {'train': set(grupos[:corte_treino]),
                    'val': set(grupos[corte_treino:corte_validacao]),
                    'test': set(grupos[corte_validacao:])}
PASTA_FINAL.mkdir(parents=True, exist_ok=True)
por_split: dict[str, list[dict[str, Any]]] = {}
for nome, bases in grupos_por_split.items():
    por_split[nome] = [r for r in registros if r['base_id'] in bases]
    destino = PASTA_FINAL / f'{nome}.jsonl'
    with destino.open('w', encoding='utf-8') as arquivo:
        for registro in por_split[nome]: arquivo.write(serializar_jsonl(registro) + '\n')

tamanho = min(int(CONFIG['review_sample_size']), len(registros))
revisao = random.sample(registros, tamanho)
linhas_csv = [{'id': r['id'], 'base_id': r['base_id'], 'erro_tipo': r['erro_tipo'],
               'problema': r['entrada']['duvida'], 'diagnostico': r['saida']['dificuldade'],
               'acao': r['saida']['acao'], 'chosen': r['dpo']['chosen'],
               'rejected': r['dpo']['rejected'], 'aprovado/reprovado': '', 'comentário': ''}
              for r in revisao]
pd.DataFrame(linhas_csv).to_csv(PASTA_FINAL / 'amostra_revisao.csv', index=False, encoding='utf-8-sig')
print('Grupos:', n, '| registros:', len(registros), '| saída:', PASTA_FINAL)

Grupos: 99 | registros: 99 | saída: c:\Users\maximus.nascimento\Downloads\calma-ai\notebooks\eja_diagnosticador_dataset\data\final


Com poucos grupos, validação ou teste podem ficar vazios; o pipeline não duplica grupos para preencher essas partes. Para avaliação útil, gere mais problemas-base e confira os tamanhos antes de usar os arquivos.

In [2]:
conjuntos_base = {nome: {r['base_id'] for r in dados} for nome, dados in por_split.items()}
assert conjuntos_base['train'].isdisjoint(conjuntos_base['val'])
assert conjuntos_base['train'].isdisjoint(conjuntos_base['test'])
assert conjuntos_base['val'].isdisjoint(conjuntos_base['test'])
assert sum(len(v) for v in por_split.values()) == len(registros)
print('Resumo por split:', {k: {'registros': len(v), 'problemas_base': len(conjuntos_base[k])} for k, v in por_split.items()})
print('CSV revisão:', len(linhas_csv), 'linhas | colunas:', list(pd.DataFrame(linhas_csv).columns))
print('Amostra de revisão:')
display(pd.DataFrame(linhas_csv).head(3))
print('JSONL de amostra:')
for nome in ('train', 'val', 'test'):
    caminho = PASTA_FINAL / f'{nome}.jsonl'
    assert caminho.exists()
    print(nome, '=>', caminho.stat().st_size, 'bytes')

Resumo por split: {'train': {'registros': 79, 'problemas_base': 79}, 'val': {'registros': 10, 'problemas_base': 10}, 'test': {'registros': 10, 'problemas_base': 10}}
CSV revisão: 99 linhas | colunas: ['id', 'base_id', 'erro_tipo', 'problema', 'diagnostico', 'acao', 'chosen', 'rejected', 'aprovado/reprovado', 'comentário']
Amostra de revisão:


,id,base_id,erro_tipo,problema,diagnostico,acao,chosen,rejected,aprovado/reprovado,comentário
0,train:25:ERRO_CALCULO,train:25,ERRO_CALCULO,Estou resolvendo uma situação de salário e tra...,Precisão do cálculo aritmético.,DAR_DICA,"Na situação de salário e trabalho, confira a r...",A resposta completa é 110.,,
1,train:38:PASSO_OMITIDO,train:38,PASSO_OMITIDO,Estou resolvendo uma situação de salário e tra...,Encadeamento incompleto do raciocínio.,VALIDAR_PARCIALMENTE,"No contexto de salário e trabalho, você já org...",A resposta completa é 9.,,
2,train:35:SEM_ERRO,train:35,SEM_ERRO,Estou resolvendo uma situação de feira e compr...,Sem dificuldade observada nas contas verificadas.,PROPOR_EXERCICIO,"No contexto de feira e compras, sua estratégia...",A resposta completa é 36.,,


JSONL de amostra:
train => 86661 bytes
val => 10889 bytes
test => 10815 bytes
